In [1]:
import pandas as pd
import re
import unicodedata
from rapidfuzz import process, fuzz
from collections import defaultdict

# ==============================================================================
# CONFIGURATION: File Names
# ==============================================================================
SEARCH_TERMS_FILE = 'search_terms.csv'
VIDEO_REPORT_FILE = 'video_report.csv'
GCDM_REPORT_FILE = 'GCDM_report.csv'
OUTPUT_FILE = 'Processed_Video_Report_with_ISRC.csv'

In [2]:
# ==============================================================================
# PART 1: Load Terms & Categorize Videos
# ==============================================================================
terms_df = pd.read_csv(SEARCH_TERMS_FILE)
def get_terms(series):
    terms = series.dropna().astype(str).str.lower().str.strip().tolist()
    return [t for t in terms if t != '' and t != 'nan']

live_terms = get_terms(terms_df['Live Performance'])
music_video_terms = get_terms(terms_df['Music Video'])
lyric_terms = get_terms(terms_df['Lyric Video'])
promo_terms = get_terms(terms_df['Promotional Video'])
visualizer_terms = get_terms(terms_df['Visualizer'])
still_image_terms = get_terms(terms_df['Still Image w/ Audio'])
karaoke_terms = get_terms(terms_df['Karaoke'])
vertical_terms = get_terms(terms_df['Vertical Video'])

df_cat = pd.read_csv(VIDEO_REPORT_FILE)
df_cat = df_cat.drop_duplicates(subset=['VIDEO_ID'], keep='first')

# HELPER: Safely parse different time formats into total minutes
def get_total_minutes(time_val):
    t_str = str(time_val).strip()
    if not t_str or t_str.lower() == 'nan':
        return 0.0
    try:
        if ':' in t_str:
            parts = t_str.split(':')
            if len(parts) == 3:  return int(parts[0]) * 60 + int(parts[1]) + int(parts[2]) / 60.0
            elif len(parts) == 2: return int(parts[0]) + int(parts[1]) / 60.0
        else:
            return float(t_str) / 60.0
    except ValueError:
        return 0.0
    return 0.0

def categorize_video(row):
    title = str(row.get('TITLE', '')).lower() 
    
    if '#shorts' in title or '#short' in title: return 'YouTube Short'
    if any(keyword in title for keyword in vertical_terms): return 'Vertical Video'
    if any(keyword in title for keyword in karaoke_terms): return 'Karaoke'
    if any(keyword in title for keyword in live_terms): return 'Live Performance'
    if any(keyword in title for keyword in lyric_terms): return 'Lyric Video'
    if any(keyword in title for keyword in visualizer_terms): return 'Visualizer'
    if any(keyword in title for keyword in promo_terms): return 'Promotional Video'
    if any(keyword in title for keyword in still_image_terms): return 'Still Image w/ Audio'
    if any(keyword in title for keyword in music_video_terms): return 'Music Video'
    if re.search(r'\(\d{4}\)', title): return 'Live Performance'
    
    # 10-Minute Duration Check for Uncategorized Videos
    time_value = row.get('TIME', '0')
    if get_total_minutes(time_value) > 10.0:
        return 'Promotional Video'
        
    return 'Uncategorized'

category_series = df_cat.apply(categorize_video, axis=1)
df_cat.insert(2, 'Video Category', category_series)

In [3]:
# ==============================================================================
# PART 2: GCDM Prep & Normalization (Search Engine Indexed)
# ==============================================================================
print("Loading GCDM Report...")
df_gcdm = pd.read_csv(GCDM_REPORT_FILE, engine='python', on_bad_lines='skip')

df_gcdm_clean = df_gcdm[['TITLE', 'ARTIST_DISPLAY_TITLE', 'ISRC', 'TYPE', 'SUB_TYPE']].dropna(subset=['TITLE', 'ISRC']).copy()
df_gcdm_clean['ARTIST_DISPLAY_TITLE'] = df_gcdm_clean['ARTIST_DISPLAY_TITLE'].fillna('')
df_gcdm_clean['SUB_TYPE'] = df_gcdm_clean['SUB_TYPE'].fillna('Unknown')

type_priority = {
    'Video (Music Work)': 1,
    'Video (Non Music Work)': 2,
    'Sound Recording (Music Work)': 3
}
df_gcdm_clean['priority'] = df_gcdm_clean['TYPE'].map(type_priority).fillna(99)
df_gcdm_clean = df_gcdm_clean.sort_values('priority')

print("Building Lookups...")
gcdm_isrc_lookup = {}
for isrc, subtype in zip(df_gcdm_clean['ISRC'], df_gcdm_clean['SUB_TYPE']):
    isrc_str = str(isrc).strip()
    if isrc_str not in gcdm_isrc_lookup:
        gcdm_isrc_lookup[isrc_str] = subtype
valid_gcdm_isrcs = set(gcdm_isrc_lookup.keys())

def clean_and_normalize(t):
    if not isinstance(t, str): return ""
    t = t.lower()
    t = unicodedata.normalize('NFKD', t)
    t = "".join([c for c in t if not unicodedata.combining(c)])
    t = re.sub(r'\(.*?\)', '', t)
    t = re.sub(r'\[.*?\]', '', t)
    t = re.sub(r'[^\w\s\-\|]', '', t)
    return t.strip()

df_gcdm_clean['norm_title'] = [clean_and_normalize(t) for t in df_gcdm_clean['TITLE']]
df_gcdm_clean['norm_artist'] = [clean_and_normalize(a) for a in df_gcdm_clean['ARTIST_DISPLAY_TITLE']]

gcdm_dict_norm = {}
for t, isrc, sub in zip(df_gcdm_clean['norm_title'], df_gcdm_clean['ISRC'], df_gcdm_clean['SUB_TYPE']):
    if t and t not in gcdm_dict_norm:
        gcdm_dict_norm[t] = (isrc, sub)
gcdm_titles_unique = list(gcdm_dict_norm.keys())

two_factor_dict = {}
two_factor_strings = {}
ignore_artists = {'los', 'las', 'the', 'el', 'la', ''}

for a, t, isrc, sub in zip(df_gcdm_clean['norm_artist'], df_gcdm_clean['norm_title'], df_gcdm_clean['ISRC'], df_gcdm_clean['SUB_TYPE']):
    if len(a) > 3 and len(t) > 3 and a not in ignore_artists:
        if (a, t) not in two_factor_dict:
            two_factor_dict[(a, t)] = (isrc, sub)
        combined_str = f"{a} {t}"
        if combined_str not in two_factor_strings:
            two_factor_strings[combined_str] = (isrc, sub)

two_factor_string_list = list(two_factor_strings.keys())

def get_yt_candidates_norm(title):
    t = clean_and_normalize(title)
    t = re.sub(r'video oficial|official video|lyric video|music video|audio oficial|cover audio|en vivo|live', '', t)
    t = re.sub(r'\bft\.?\b', '-', t)
    t = re.sub(r'\bfeat\.?\b', '-', t)
    t = re.sub(r'\bx\b', '-', t)
    t = t.strip()
    
    candidates = [t]
    split_parts = []
    
    if '-' in t:
        parts = [p.strip() for p in t.split('-') if p.strip()]
        if len(parts) >= 2:
            candidates.extend([parts[1], parts[-1], parts[0]])
            split_parts = parts
            
    if '|' in t:
        parts = [p.strip() for p in t.split('|') if p.strip()]
        if len(parts) >= 2:
            candidates.extend([parts[0], parts[1]])
            if not split_parts: split_parts = parts

    seen = set()
    deduped_candidates = [c for c in candidates if c and len(c) > 2 and not (c in seen or seen.add(c))]
    return deduped_candidates, t, split_parts

print("Building Inverted Word Indexes (This will take a few seconds)...")
word_index = defaultdict(set)
for t in gcdm_titles_unique:
    for word in t.split():
        if len(word) >= 3:
            word_index[word].add(t)

tf_word_index = defaultdict(set)
for tf_str in two_factor_string_list:
    for word in tf_str.split():
        if len(word) >= 3:
            tf_word_index[word].add(tf_str)
            
print("Indexes built successfully!")

Loading GCDM Report...
Building Lookups...
Building Inverted Word Indexes (This will take a few seconds)...
Indexes built successfully!


In [4]:
# ==============================================================================
# PART 3: Execute Waterfall Match Logic
# ==============================================================================
print("Starting High-Speed Index Matching Process...")
final_isrcs = []
final_subtypes = []  
match_sources = []
yt_isrc_missing_flags = []

records = df_cat.to_dict('records')

for i, row in enumerate(records):
    # Progress tracker
    if i > 0 and i % 5000 == 0:
        print(f"Processed {i} out of {len(records)} videos...", flush=True)
        
    found_isrc = None
    found_subtype = None  
    match_source = "No Match"
    yt_isrc_missing = "No YT ISRC Provided"
    
    yt_isrc_raw = str(row.get('ISRC', '')).strip()
    yt_claim_raw = str(row.get('CLAIM_CUSTOM_ID', '')).strip()
    
    def get_potential_ids(val):
        if not val or val.lower() == 'nan': return []
        return [item.strip() for item in val.split(',') if item.strip()]

    potential_ids = get_potential_ids(yt_isrc_raw) + get_potential_ids(yt_claim_raw)
    
    # STEP 1: Strict YouTube Validation
    if potential_ids:
        valid_found = next((item for item in potential_ids if item in valid_gcdm_isrcs), None)
        if valid_found:
            found_isrc = valid_found
            found_subtype = gcdm_isrc_lookup[valid_found] 
            match_source = "YouTube ISRC Matches GCDM ISRC"
            yt_isrc_missing = "No"
        else:
            yt_isrc_missing = "Yes - Flagged"
            
    yt_title = str(row.get('TITLE', ''))
    candidates, full_norm_title, split_parts = get_yt_candidates_norm(yt_title)
    
    yt_words = set(w for w in full_norm_title.split() if len(w) >= 3)
    
    # STEP 2: Exact Title Match
    if found_isrc is None:
        for c in candidates:
            if c in gcdm_dict_norm:
                found_isrc, found_subtype = gcdm_dict_norm[c] 
                match_source = "GCDM Exact Title Match"
                break
                
    # STEP 3: Two-Factor Match
    if found_isrc is None:
        matched_via_split = False
        
        # 3A: Fast Split Check
        if len(split_parts) >= 2:
            for j in range(len(split_parts)-1):
                part1 = split_parts[j]
                part2 = split_parts[j+1]
                if (part1, part2) in two_factor_dict:
                    found_isrc, found_subtype = two_factor_dict[(part1, part2)]
                    match_source = "GCDM Two-Factor Match (Exact Split)"
                    matched_via_split = True
                    break
                elif (part2, part1) in two_factor_dict:
                    found_isrc, found_subtype = two_factor_dict[(part2, part1)]
                    match_source = "GCDM Two-Factor Match (Exact Split)"
                    matched_via_split = True
                    break
                    
        # 3B: Token Check with Inverted Index
        if not matched_via_split and yt_words:
            tf_candidates = set()
            for w in yt_words:
                if w in tf_word_index:
                    tf_candidates.update(tf_word_index[w])
            
            if tf_candidates:
                best_tf = process.extractOne(full_norm_title, tf_candidates, scorer=fuzz.token_set_ratio, score_cutoff=95)
                if best_tf:
                    tf_match_string = best_tf[0]
                    found_isrc, found_subtype = two_factor_strings[tf_match_string]
                    match_source = "GCDM Two-Factor Match (Word Containment)"
                
    # STEP 4: Fuzzy Match with Inverted Index
    if found_isrc is None:
        for c in candidates:
            if len(c) > 4: 
                c_words = set(w for w in c.split() if len(w) >= 3)
                fuzzy_candidates = set()
                for w in c_words:
                    if w in word_index:
                        fuzzy_candidates.update(word_index[w])
                
                if fuzzy_candidates:
                    best_match = process.extractOne(c, fuzzy_candidates, scorer=fuzz.ratio, score_cutoff=88)
                    if best_match:
                        match_title = best_match[0] 
                        found_isrc, found_subtype = gcdm_dict_norm[match_title]
                        match_source = "GCDM Fuzzy Match (>88% Similar)"
                        break
                    
    final_isrcs.append(found_isrc)
    final_subtypes.append(found_subtype)
    match_sources.append(match_source)
    yt_isrc_missing_flags.append(yt_isrc_missing)

Starting High-Speed Index Matching Process...
Processed 5000 out of 211897 videos...
Processed 10000 out of 211897 videos...
Processed 15000 out of 211897 videos...
Processed 20000 out of 211897 videos...
Processed 25000 out of 211897 videos...
Processed 30000 out of 211897 videos...
Processed 35000 out of 211897 videos...
Processed 40000 out of 211897 videos...
Processed 45000 out of 211897 videos...
Processed 50000 out of 211897 videos...
Processed 55000 out of 211897 videos...
Processed 60000 out of 211897 videos...
Processed 65000 out of 211897 videos...
Processed 70000 out of 211897 videos...
Processed 75000 out of 211897 videos...
Processed 80000 out of 211897 videos...
Processed 85000 out of 211897 videos...
Processed 90000 out of 211897 videos...
Processed 95000 out of 211897 videos...
Processed 100000 out of 211897 videos...
Processed 105000 out of 211897 videos...
Processed 110000 out of 211897 videos...
Processed 115000 out of 211897 videos...
Processed 120000 out of 211897 

In [5]:
# ==============================================================================
# PART 4: Append, Reorder & Export
# ==============================================================================
df_cat['Matched_ISRC'] = final_isrcs
df_cat['GCDM_Sub_Type'] = final_subtypes  
df_cat['ISRC_Match_Source'] = match_sources
df_cat['YT_ISRC_Missing_From_GCDM'] = yt_isrc_missing_flags

# Reorder columns so priority ones are at the front (A-D logic)
leading_cols = [
    'Video Category', 
    'Matched_ISRC',
    'GCDM_Sub_Type', 
    'ISRC_Match_Source', 
    'YT_ISRC_Missing_From_GCDM'
]
remaining_cols = [col for col in df_cat.columns if col not in leading_cols]
df_cat = df_cat[leading_cols + remaining_cols]

# Export
df_cat.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"Exported successfully to: {OUTPUT_FILE}")

Exported successfully to: Processed_Video_Report_with_ISRC.csv
